# try the not renamed event log instead 

- import libraries:

In [1]:
import pm4py # process mining library
import pandas as pd # data manipulation library
import os # operating system library for file handling
import matplotlib.pyplot as plt # plotting library for visualizations
import numpy as np # numerical computing library for data analysis
import networkx as nx # library for graph analysis and visualization
import math # mathematical functions for calculations

- load only completed and relabled events 

In [2]:
df = pd.read_csv("../data/complete_log.csv", low_memory=False)
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], format='mixed', utc=True)
event_log = pm4py.format_dataframe(df, case_id='case:concept:name', activity_key='concept:name', timestamp_key='time:timestamp')


### Define own Simplicity metrics

In the book [3], the following metrics are mentioned as candidates for simplicity metrics by analyzing the complexity of the underlying graph:
- Size
- Diameter
- Density
- Connectivity
- Node Degree

Given that the graph functions as a process model, additional domain-specific metrics can be applied:

- Sequentiality: Gateway arcs divided by total arcs.
- Structuredness: Proportion of reducible, well-structured parts in the model.
- Depth: Average or max depth of split/join constructs.
- Gateway Mismatch: Difference between input and output arcs for connected gateways.
- Gateway Heterogeneity: Entropy of gateway types used.
- Control Flow Complexity: Sum of choices based on split types and outgoing arcs.
- Cyclicity: Ratio of nodes in cycles to total nodes.
- Token Splits: Number of concurrent threads from AND/OR-splits.


[3] Josep Carmona, Boudewijn F. van Dongen, Andreas Solti, and Matthias Weidlich. *Conformance Checking - Relating Processes and Models*. Springer, 2018, pp. 223–224.

- a first simple simplicity metrics is the size (arcs and nodes) that is implemented in the following way:

In [3]:
def calculate_node_size(net):
    """
    Calculates the total number of nodes in a pm4py Petri net.
    Nodes = Places + Transitions
    """
    num_places = len(net.places)
    num_transitions = len(net.transitions)
    
    return num_places + num_transitions

def calculate_arc_size(net):
    """
    Calculates the total number of arcs (edges) in a pm4py Petri net.
    """
    return len(net.arcs)

- another metric that makes sense in general is the cyclicity of the net
- implementation:

In [4]:
def calculate_cyclicity(net):
    """
    Calculates the cyclicity of a pm4py Petri net.
    Cyclicity = (number of nodes within cycles) / (total number of nodes)
    
    :param net: pm4py Petri net object
    :return: Float representing the cyclicity metric (between 0.0 and 1.0)
    """
    # 1. Calculate total number of nodes (Places + Transitions)
    total_nodes = len(net.places) + len(net.transitions)
    
    if total_nodes == 0:
        return 0.0
        
    # 2. Build a NetworkX Directed Graph
    G = nx.DiGraph()
    
    # Add all places and transitions as nodes
    for place in net.places:
        G.add_node(place)
    for transition in net.transitions:
        G.add_node(transition)
        
    # Add all arcs as directed edges
    for arc in net.arcs:
        G.add_edge(arc.source, arc.target)
        
    # 3. Identify nodes in cycles
    # A node is in a cycle if it is part of a Strongly Connected Component (SCC)
    # with more than 1 node, or if it is a single node with a self-loop.
    nodes_in_cycles = set()
    
    for scc in nx.strongly_connected_components(G):
        if len(scc) > 1:
            # All nodes in an SCC of size > 1 are part of a cycle
            nodes_in_cycles.update(scc)
        elif len(scc) == 1:
            # Check for self-loops for single-node components
            node = list(scc)[0]
            if G.has_edge(node, node):
                nodes_in_cycles.add(node)
                
    # 4. Calculate the cyclicity ratio
    cyclicity = len(nodes_in_cycles) / total_nodes
    
    return cyclicity

### process discovery 

In [ ]:
output_dir = "../petrinets/inductive_fine_grid_search_top610"
os.makedirs(output_dir, exist_ok=True)
results_dir = "../results/inductive"
os.makedirs(results_dir, exist_ok=True)

k_variants = 610
print(f"Filtering for top {k_variants} variants...")
filtered_log = pm4py.filter_variants_top_k(event_log, k_variants)

noise_thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99]
results_list = []

print(f"Running Inductive Miner fine grid search on {len(noise_thresholds)} combinations...")
for noise in noise_thresholds:
    net, im, fm = pm4py.discover_petri_net_inductive(filtered_log, noise_threshold=noise)
    
    model_name = f"ind_top610_noise_{noise:.2f}"
    pnml_path = os.path.join(output_dir, f"{model_name}.pnml")
    pdf_path = os.path.join(output_dir, f"{model_name}.pdf")
    
    pm4py.write_pnml(net, im, fm, pnml_path)
    pm4py.save_vis_petri_net(net, im, fm, pdf_path)
    
    fitness = pm4py.algo.evaluation.replay_fitness.variants.token_replay.apply(event_log, net, im, fm)
    precision = pm4py.algo.evaluation.precision.variants.etconformance_token.apply(event_log, net, im, fm)
    generalization = pm4py.algo.evaluation.generalization.variants.token_based.apply(event_log, net, im, fm)
    
    perfect_fit_traces = fitness.get('perc_fit_traces', 0)
    
    results_list.append({
        'Model Name': model_name,
        'Noise Thresh': noise,
        'Perfect Fitting Traces (%)': round(perfect_fit_traces, 2),
        'Average Trace Fitness': fitness['log_fitness'],
        'Precision': precision,
        'Generalization': generalization,
        'Node Size': calculate_node_size(net),
        'Arc Size': calculate_arc_size(net),
        'Cyclicity': calculate_cyclicity(net)
    })
    print(f"Done: {model_name}")

grid_search_df = pd.DataFrame(results_list)
grid_search_df = grid_search_df.sort_values(by=['Average Trace Fitness', 'Precision'], ascending=False)
grid_search_df.to_csv(os.path.join(results_dir, "inductive_grid_search_results_top610.csv"), index=False)
grid_search_df

Filtering for top 610 variants...
Running Inductive Miner fine grid search on 11 combinations...


replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/27816 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

Done: ind_top610_noise_0.00


...

replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/27816 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

Done: ind_top610_noise_0.99


,Model Name,Noise Thresh,Perfect Fitting Traces (%),Average Trace Fitness,Precision,Generalization,Node Size,Arc Size,Cyclicity
0,ind_top610_noise_0.00,0.00,99.64,0.999888,0.327184,0.971401,133,172,0.819549
1,ind_top610_noise_0.10,0.10,61.65,0.981579,0.668006,0.941303,88,108,0.522727
2,ind_top610_noise_0.20,0.20,68.71,0.977517,0.624337,0.935563,99,122,0.828283
3,ind_top610_noise_0.30,0.30,52.94,0.969845,0.628345,0.918144,99,122,0.828283
4,ind_top610_noise_0.40,0.40,29.89,0.948163,0.640677,0.915394,97,120,0.835052
5,ind_top610_noise_0.50,0.50,14.87,0.925454,0.643103,0.913903,96,118,0.833333
6,ind_top610_noise_0.60,0.60,1.90,0.883727,0.696685,0.890768,63,76,0.396825
7,ind_top610_noise_0.70,0.70,3.42,0.873065,0.636526,0.827384,62,76,0.387097
9,ind_top610_noise_0.90,0.90,0.16,0.848500,0.905956,0.881722,42,46,0.428571
10,ind_top610_noise_0.99,0.99,0.16,0.848500,0.905956,0.881722,42,46,0.428571
